Tiền xử lý dữ liệu House Price Prediction

Mục tiêu:
- Kiểm tra dữ liệu trùng lặp và ngoại lệ.
- Bỏ cột Id nếu tồn tại.
- Chuyển các cột phân loại bằng one-hot encoding.
- Chuẩn hóa các cột số cho SVM.
- Chia dữ liệu thành tập huấn luyện và kiểm tra theo tỷ lệ 80/20.
- Xây dựng hàm đánh giá trả về R², RMSE, MAE và thời gian.

Cột mục tiêu: price.

# Import thư viện

In [ ]:
from time import perf_counter

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVR

# Load dataset

In [ ]:
df = pd.read_csv("../Data/data.csv")

print("Kích thước:", df.shape)
display(df.head())

Kích thước: (4600, 18)


,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,yr_built,yr_renovated,street,city,statezip,country
0,2014-05-02 00:00:00,313000.0,3.0,1.50,1340,7912,1.5,0,0,3,1340,0,1955,2005,18810 Densmore Ave N,Shoreline,WA 98133,USA
1,2014-05-02 00:00:00,2384000.0,5.0,2.50,3650,9050,2.0,0,4,5,3370,280,1921,0,709 W Blaine St,Seattle,WA 98119,USA
2,2014-05-02 00:00:00,342000.0,3.0,2.00,1930,11947,1.0,0,0,4,1930,0,1966,0,26206-26214 143rd Ave SE,Kent,WA 98042,USA
3,2014-05-02 00:00:00,420000.0,3.0,2.25,2000,8030,1.0,0,0,4,1000,1000,1963,0,857 170th Pl NE,Bellevue,WA 98008,USA
4,2014-05-02 00:00:00,550000.0,4.0,2.50,1940,10500,1.0,0,0,4,1140,800,1976,1992,9105 170th Ave NE,Redmond,WA 98052,USA


## Checking missing, remove ID

In [ ]:
df = df.drop(columns=["Id", "id"], errors="ignore")

display(df.isna().sum().to_frame("missing_count"))

duplicate_count = df.duplicated().sum()
print("Số dòng trùng:", duplicate_count)

df = df.drop_duplicates().copy()

print("Kích thước sau khi xử lý trùng:", df.shape)

,missing_count
date,0
price,0
bedrooms,0
bathrooms,0
sqft_living,0
sqft_lot,0
floors,0
waterfront,0
view,0
condition,0


Số dòng trùng: 0
Kích thước sau khi xử lý trùng: (4600, 18)


In [ ]:
df["price"] = pd.to_numeric(df["price"], errors="coerce")

valid_target = df["price"].notna() & np.isfinite(df["price"])

print("Số dòng thiếu hoặc sai mục tiêu:", (~valid_target).sum())

df = df.loc[valid_target].copy()

if "date" in df.columns:
    dates = pd.to_datetime(df["date"], errors="coerce")

    df["sale_year"] = dates.dt.year
    df["sale_month"] = dates.dt.month

df = df.drop(
    columns=["date", "street", "country"],
    errors="ignore",
)

Số dòng thiếu hoặc sai mục tiêu: 0


# Split train/test 80/20

In [ ]:
X = df.drop(columns=["price"])
y = df["price"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

features = {
    "X_train": X_train,
    "X_test": X_test,
    "y_train": y_train,
    "y_test": y_test,
}

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (3680, 16)
X_test: (920, 16)
y_train: (3680,)
y_test: (920,)


# checking outliner

In [ ]:
def check_outliers(data, columns):
    results = []

    for column in columns:
        values = data[column].dropna()

        q1 = values.quantile(0.25)
        q3 = values.quantile(0.75)
        iqr = q3 - q1

        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

        mask = (values < lower) | (values > upper)

        results.append({
            "column": column,
            "lower": lower,
            "upper": upper,
            "outlier_count": int(mask.sum()),
            "outlier_percent": (
                float(mask.mean() * 100) if len(values) else 0.0
            ),
        })

    return pd.DataFrame(results)

In [ ]:
train_data = X_train.copy()
train_data["price"] = y_train

outlier_report = check_outliers(
    train_data,
    ["price", "sqft_living", "sqft_lot"],
)

display(outlier_report)

,column,lower,upper,outlier_count,outlier_percent
0,price,-178656.25,1156393.75,185,5.027174
1,sqft_living,-268.75,4341.25,104,2.826087
2,sqft_lot,-4071.25,20130.75,429,11.657609


In [ ]:
def build_preprocessor(X_train, scale=True):
    numeric_columns = X_train.select_dtypes(
        include="number"
    ).columns.tolist()

    categorical_columns = X_train.select_dtypes(
        exclude="number"
    ).columns.tolist()

    numeric_steps = [
        ("imputer", SimpleImputer(strategy="median")),
    ]

    if scale:
        numeric_steps.append(
            ("scaler", StandardScaler())
        )

    numeric_pipeline = Pipeline(numeric_steps)

    categorical_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="most_frequent"),
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
        ),
    ])

    return ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_columns),
        (
            "categorical",
            categorical_pipeline,
            categorical_columns,
        ),
    ])

In [ ]:
def evaluate(model, features):
    estimator = clone(model)

    start = perf_counter()

    estimator.fit(
        features["X_train"],
        features["y_train"],
    )

    after_fit = perf_counter()

    predictions = estimator.predict(features["X_test"])

    after_predict = perf_counter()

    return {
        "R2": float(
            r2_score(features["y_test"], predictions)
        ),
        "RMSE": float(
            np.sqrt(
                mean_squared_error(
                    features["y_test"],
                    predictions,
                )
            )
        ),
        "MAE": float(
            mean_absolute_error(
                features["y_test"],
                predictions,
            )
        ),
        "train_time_seconds": after_fit - start,
        "predict_time_seconds": after_predict - after_fit,
        "total_time_seconds": after_predict - start,
    }

# Test SVM

In [ ]:
svm_pipeline = Pipeline([
    (
        "preprocess",
        build_preprocessor(X_train, scale=True),
    ),
    (
        "model",
        SVR(kernel="rbf", C=10, epsilon=0.1),
    ),
])

svm_model = TransformedTargetRegressor(
    regressor=svm_pipeline,
    transformer=StandardScaler(),
)

result = evaluate(svm_model, features)

display(pd.DataFrame([result], index=["SVR"]))

,R2,RMSE,MAE,train_time_seconds,predict_time_seconds,total_time_seconds
SVR,0.0544,982021.696059,153218.964834,1.897152,0.270913,2.168066


In [ ]:
preprocessor = build_preprocessor(X_train, scale=True)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Train sau xử lý:", X_train_processed.shape)
print("Test sau xử lý:", X_test_processed.shape)

feature_names = preprocessor.get_feature_names_out()
print(feature_names)

Train sau xử lý: (3680, 134)
Test sau xử lý: (920, 134)
['numeric__bedrooms' 'numeric__bathrooms' 'numeric__sqft_living'
 'numeric__sqft_lot' 'numeric__floors' 'numeric__waterfront'
 'numeric__view' 'numeric__condition' 'numeric__sqft_above'
 'numeric__sqft_basement' 'numeric__yr_built' 'numeric__yr_renovated'
 'numeric__sale_year' 'numeric__sale_month' 'categorical__city_Algona'
 'categorical__city_Auburn' 'categorical__city_Beaux Arts Village'
 'categorical__city_Bellevue' 'categorical__city_Black Diamond'
 'categorical__city_Bothell' 'categorical__city_Burien'
 'categorical__city_Carnation' 'categorical__city_Clyde Hill'
 'categorical__city_Covington' 'categorical__city_Des Moines'
 'categorical__city_Duvall' 'categorical__city_Enumclaw'
 'categorical__city_Fall City' 'categorical__city_Federal Way'
 'categorical__city_Issaquah' 'categorical__city_Kenmore'
 'categorical__city_Kent' 'categorical__city_Kirkland'
 'categorical__city_Lake Forest Park' 'categorical__city_Maple Valley'
 '